# YOLO11n · RUOD-R · Google Colab 训练

对应原文件：`scripts/yolo11n_e100_i960.py`。训练轮数 **100**，图像尺寸 **960**，batch **8**。

本 Notebook 已内置台账、FPS 函数，仅上传此 `.ipynb` 和数据集即可运行，无需上传 `scripts`。数据集已在本地检查，此处直接使用。

1. 在 Colab 打开 Notebook，在“运行时 → 更改运行时类型”中选择 GPU。
2. 将 `RUOD-R-YOLO-Split.zip`（或数据集目录）上传 Google Drive。
3. 修改第 2 节的 `DATA_SOURCE` 和 `DRIVE_PROJECT_ROOT`，再依次执行全部单元格。
4. 数据集要求包含 `images/train`、`images/val`、`images/test` 与对应的 `labels/...`。类别编号保持原项目的 10 类映射。

结果直接写入 Drive 的 `runs/` 和 `records/`。`patience=10` 可能使训练早于设定轮数结束。显存不足时先减小 `BATCH_SIZE`，并在比较实验时记录调整。

## 1. 安装依赖并导入

直接使用 `%pip install ultralytics` 安装，实验环境由你统一管理。环境快照会记录实际使用的依赖版本。
如果 Colab 提示重启运行时，请重启后从此处重新执行。

In [ ]:
%pip install ultralytics

In [ ]:
from pathlib import Path
import csv
import json
import os
import random
import shutil
import stat
import statistics
import subprocess
import sys
import tempfile
import time
import zipfile

import torch
import ultralytics
import yaml
from ultralytics import YOLO
from IPython.display import Image, display

## 2. 挂载 Drive、设置路径

Colab 无法直接读取本机 `E:` 盘；请先把数据上传到自己的 Drive。`DRIVE_PROJECT_ROOT` 不必已有项目代码。
默认将数据放在运行时本地磁盘以减少 Drive 小文件读取，模型和记录保存到 Drive。
`COPY_DATA_TO_LOCAL=False` 可直接使用 Drive 目录；ZIP 始终需要本地解压。

In [ ]:
# 修改这两个 Drive 路径即可开始；DATA_SOURCE 支持目录或 .zip。
USE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/underwater_object_detection"
DATA_SOURCE = "/content/drive/MyDrive/datasets/RUOD-R-YOLO-Split.zip"
COPY_DATA_TO_LOCAL = True
LOCAL_DATA_DIR = "/content/datasets/ruod_r"

# 通常保持为空；中断后填本次实验的完整 last.pt 路径以继续训练。
# 例：/content/drive/MyDrive/underwater_object_detection/runs/实验名/weights/last.pt
RESUME_CHECKPOINT = ""
REQUIRE_GPU = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = Path(DRIVE_PROJECT_ROOT if USE_DRIVE else "/content/underwater_object_detection")
PROJECT_DIR = PROJECT_ROOT / "runs"
RUNTIME_ROOT = Path("/content/underwater_object_detection_colab")
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(RUNTIME_ROOT)
print("实验结果与台账目录:", PROJECT_ROOT)

## 3. 检查训练环境

In [ ]:
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Ultralytics:", ultralytics.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("请在 Colab 的‘更改运行时类型’中选择 GPU 后重新运行。")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print("训练设备:", gpu_name)
if torch.cuda.is_available():
    print(f"GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")

## 4. 准备数据并生成 Colab 数据配置

ZIP 内允许包一层目录。更换或更新数据后请更换 `LOCAL_DATA_DIR`；不会自动删除已有缓存。
自动生成的 `data_colab.yaml` 位于 Colab 临时目录，不依赖本地项目的绝对路径。

In [ ]:
def locate_dataset(folder):
    """接受 zip 内直接放 images/labels，也接受外面多包一层目录。"""
    candidates = [folder] + [p.parent for p in folder.rglob("images") if p.is_dir()]
    valid = sorted({p.resolve() for p in candidates
                    if (p / "images" / "train").is_dir()
                    and (p / "labels" / "train").is_dir()})
    if len(valid) != 1:
        raise ValueError(f"需要唯一的数据集根目录，实际找到 {len(valid)} 个: {valid}")
    return valid[0]


def prepare_dataset(source, local_dir, copy_local=True):
    source = Path(source).expanduser()
    if not source.exists():
        raise FileNotFoundError(f"请上传数据集并修改 DATA_SOURCE，当前路径不存在: {source}")
    if source.is_dir() and not copy_local:
        return locate_dataset(source)
    if source.is_file() and source.suffix.lower() != ".zip":
        raise ValueError("数据源仅支持文件夹或 ZIP 压缩包。")
    local_dir = Path(local_dir).resolve()
    # 防止将目录复制到自身内部，或覆盖一个不相关的缓存。
    if source.is_dir() and (local_dir == source.resolve() or source.resolve() in local_dir.parents):
        raise ValueError("LOCAL_DATA_DIR 必须位于源数据目录之外。")
    fingerprint = {"source": str(source.resolve()), "mtime_ns": source.stat().st_mtime_ns,
                   "size": source.stat().st_size if source.is_file() else None}
    marker = local_dir / ".colab_dataset_source.json"
    if marker.is_file() and json.loads(marker.read_text()) == fingerprint:
        print("复用本次运行时的数据缓存；若目录中的数据有更新，请更换 LOCAL_DATA_DIR。")
        return locate_dataset(local_dir)
    if local_dir.exists() and any(local_dir.iterdir()):
        raise FileExistsError(f"缓存目录非空且来源不匹配，请更换 LOCAL_DATA_DIR: {local_dir}")
    local_dir.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, local_dir, dirs_exist_ok=True)
    else:
        # 先将单个 ZIP 复制到运行时，再进行大量小文件解压。
        with tempfile.TemporaryDirectory(prefix="ruod_zip_") as temp_dir:
            local_zip = Path(temp_dir) / "dataset.zip"
            shutil.copy2(source, local_zip)
            with zipfile.ZipFile(local_zip) as archive:
                for item in archive.infolist():
                    target = (local_dir / item.filename).resolve()
                    if not target.is_relative_to(local_dir):
                        raise ValueError(f"ZIP 中存在越界路径: {item.filename}")
                    if stat.S_ISLNK(item.external_attr >> 16):
                        raise ValueError(f"ZIP 中存在符号链接: {item.filename}")
                archive.extractall(local_dir)
    root = locate_dataset(local_dir)
    marker.write_text(json.dumps(fingerprint, ensure_ascii=False), encoding="utf-8")
    return root


DATA_ROOT = prepare_dataset(DATA_SOURCE, LOCAL_DATA_DIR, COPY_DATA_TO_LOCAL)
print("用于训练的数据集:", DATA_ROOT)

In [ ]:
DATA_CONFIG = {'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 10, 'names': {0: 'holothurian', 1: 'echinus', 2: 'scallop', 3: 'starfish', 4: 'fish', 5: 'corals', 6: 'diver', 7: 'cuttlefish', 8: 'turtle', 9: 'jellyfish'}}
DATA_CONFIG["path"] = str(DATA_ROOT)
DATA_YAML = RUNTIME_ROOT / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(DATA_CONFIG, allow_unicode=True, sort_keys=False), encoding="utf-8")
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
print(DATA_YAML.read_text())

## 5. 训练参数

默认值与对应训练脚本一致。修改参数后应重新开始一轮实验；断点续训请保持原模型与参数。
测试集用于最终评估，不用于选择模型或调参。

In [ ]:
EXPERIMENT_NAME = 'yolo11n_epoch100_imgsz960'
MODEL_NAME = 'YOLO11n'
PRETRAINED_WEIGHTS = 'yolo11n.pt'
DATASET_NAME = 'RUOD-R'
EPOCHS = 100
BATCH_SIZE = 8
IMG_SIZE = 960
WORKERS = 0
SEED = 42
PATIENCE = 10
OPTIMIZER = 'auto'
SAVE_SAMPLE_PREDICTIONS = True
SAMPLE_PREDICTION_COUNT = 20
PREDICTION_CONF = 0.25

## 6. 内置实验辅助函数

来自 `experiment_ledger.py`、`inference_benchmark.py` 和当前训练脚本；不需要额外模块文件。
台账字段与原项目保持一致。

In [ ]:
FIELDS = ['experiment_name', 'model', 'dataset', 'epochs', 'actual_epochs', 'best_epoch', 'batch', 'imgsz', 'optimizer', 'seed', 'parameters_M', 'gflops', 'gpu_name', 'gpu_peak_memory_gib', 'val_precision', 'val_recall', 'val_mAP50', 'val_mAP50_95', 'test_precision', 'test_recall', 'test_mAP50', 'test_mAP50_95', 'train_minutes', 'fps_mean', 'fps_std']

def save_experiment_record(record, project_root):

    save_dir = Path(project_root) / "records"
    save_dir.mkdir(parents=True, exist_ok=True)

    csv_path = save_dir / "experiment_ledger.csv"

    is_new = (
        not csv_path.exists()
        or csv_path.stat().st_size == 0
    )

    # 检查已有台账表头
    if not is_new:
        with csv_path.open(
            "r", encoding="utf-8-sig", newline=""
        ) as f:
            header = next(csv.reader(f), [])

        if header != FIELDS:
            raise ValueError(
                "现有台账字段不一致，请先备份旧 CSV。"
            )

    # 追加一条实验记录
    with csv_path.open(
        "a",
        newline="",
        encoding="utf-8-sig" if is_new else "utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=FIELDS
        )

        if is_new:
            writer.writeheader()

        writer.writerow({
            field: record.get(field, "")
            for field in FIELDS
        })

    print(f"\n实验台账已保存: {csv_path}")

In [ ]:
def print_metrics(metrics, split_name):

    print(f"\n====== {split_name} 评估结果 ======")

    print(f"Precision:     {metrics.box.mp:.4f}")
    print(f"Recall:        {metrics.box.mr:.4f}")
    print(f"mAP@0.5:       {metrics.box.map50:.4f}")
    print(f"mAP@0.5:0.95:  {metrics.box.map:.4f}")

def save_sample_predictions(model, run_dir):

    if not SAVE_SAMPLE_PREDICTIONS:
        return 0

    print("\n========== 保存预测示例 ==========")

    # 测试集图片目录
    test_dir = DATA_ROOT / "images" / "test"

    # 获取测试集图片
    test_images = sorted(
        p for p in test_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_SUFFIXES
    )

    if not test_images:
        print("测试集没有可供预测的图片。")
        return 0

    # 随机选取测试图片
    sample_count = min(
        SAMPLE_PREDICTION_COUNT,
        len(test_images)
    )

    rng = random.Random(SEED)

    sample_images = rng.sample(
        test_images,
        sample_count
    )

    # 创建预测保存目录
    save_dir = run_dir / "sample_predictions"
    save_dir.mkdir(parents=True, exist_ok=True)
    source_file = save_dir / "sources.txt"
    source_file.write_text("\n".join(str(p.resolve()) for p in sample_images), encoding="utf-8")

    # 模型预测
    results = model.predict(
        source=str(source_file),
        batch=1,
        imgsz=IMG_SIZE,
        conf=PREDICTION_CONF,
        device=DEVICE,
        stream=True,
        save=False,
        verbose=False,
    )

    saved_count = 0

    # 保存带检测框的图片
    for index, result in enumerate(results, start=1):

        image_name = Path(result.path).stem

        save_path = save_dir / (
            f"{index:02d}_{image_name}.jpg"
        )

        result.save(filename=str(save_path))

        saved_count += 1

        print(
            f"[{index}/{sample_count}] "
            f"已保存: {save_path.name}"
        )

    print(f"预测示例保存完成，共{saved_count}张。")

    return saved_count

In [ ]:
def _sync(device):
    """等待 GPU 计算完成，保证计时准确。"""

    if not torch.cuda.is_available():
        return

    if str(device).lower() == "cpu":
        return

    if isinstance(device, int) or str(device).isdigit():
        torch.cuda.synchronize(int(device))
    else:
        torch.cuda.synchronize(device)

def benchmark_fps(
    model,
    test_dir,
    imgsz=640,
    device=0,
    batch=1,
    warmup=50,
    repeats=3,
):
    """
    测量完整测试集的端到端 FPS。

    包含：图片读取、解码、预处理、模型推理、后处理。
    不包含：模型加载、预热、结果可视化和文件保存。

    返回：
        inference_fps: FPS 均值
        inference_fps_std: 多次测试的标准差
    """

    test_dir = Path(test_dir)

    if not test_dir.is_dir():
        raise FileNotFoundError(
            f"测试集目录不存在: {test_dir}"
        )

    image_paths = sorted(
        p for p in test_dir.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_SUFFIXES
    )

    if not image_paths:
        raise RuntimeError(
            f"测试集没有图片: {test_dir}"
        )

    if batch < 1 or warmup < 1 or repeats < 1:
        raise ValueError(
            "batch、warmup、repeats 必须大于 0。"
        )

    sources = [str(p) for p in image_paths]

    # 文件清单走文件加载器，使 batch 生效；避免全量图片列表驻留内存。
    with tempfile.TemporaryDirectory(prefix="fps_sources_") as temporary_dir:
        source_file = Path(temporary_dir) / "test_images.txt"
        source_file.write_text("\n".join(str(p.resolve()) for p in image_paths), encoding="utf-8")
        print("\n========== FPS 性能测试 ==========")
        print(f"测试集图片数: {len(sources)}")
        print(f"输入尺寸: {imgsz}")
        print(f"Batch Size: {batch}")
        print(f"预热次数: {warmup}")
        print(f"正式测试次数: {repeats}")

        # --------------------------------------
        # 1. 预热，不计入 FPS
        # --------------------------------------

        print("正在预热模型...")

        for _ in range(warmup):
            results = model.predict(
                source=str(image_paths[0]),
                imgsz=imgsz,
                batch=1,
                device=device,
                half=False,
                stream=True,
                save=False,
                verbose=False,
            )

            # 必须消费生成器，才能真正执行预测
            for _ in results:
                pass

        _sync(device)

        # --------------------------------------
        # 2. 重复测试完整测试集
        # --------------------------------------

        fps_values = []

        for repeat in range(repeats):

            _sync(device)
            start = time.perf_counter()

            processed = 0

            results = model.predict(
                source=str(source_file),
                imgsz=imgsz,
                batch=batch,
                device=device,
                half=False,
                stream=True,
                save=False,
                verbose=False,
            )

            for _ in results:
                processed += 1

            # 确保 GPU 运算完成后再停止计时
            _sync(device)

            elapsed = time.perf_counter() - start

            if processed != len(sources):
                raise RuntimeError(
                    f"预期处理 {len(sources)} 张图片，"
                    f"实际处理 {processed} 张。"
                )

            fps = processed / elapsed
            fps_values.append(fps)

            print(
                f"第 {repeat + 1}/{repeats} 次: "
                f"{fps:.2f} FPS"
            )

        # --------------------------------------
        # 3. 汇总结果
        # --------------------------------------

        fps_mean = statistics.mean(fps_values)

        fps_std = (
            statistics.stdev(fps_values)
            if len(fps_values) > 1
            else 0.0
        )

        print("\n========== FPS 测试结果 ==========")
        print(f"FPS Mean: {fps_mean:.2f}")
        print(f"FPS Std:  {fps_std:.2f}")

        return fps_mean, fps_std

## 7. 加载模型（新训练或恢复训练）

新训练保留 `RESUME_CHECKPOINT=""`。中断后重新执行前面的准备步骤，将该参数改为同一次实验的 `weights/last.pt`，再继续运行。
恢复只适用于尚未完成的训练；训练结束后被移除优化器状态的权重不能用 `resume=True` 继续。
续训会使用权重中保存的训练参数和原输出目录，请使用同一个 Drive 目录及同一个 Notebook。
续训后无法从新建的 early-stopping 对象可靠还原全程最佳轮次，因此台账的 `best_epoch` 留空；最佳权重仍以 `best.pt` 为准。

In [ ]:
if RESUME_CHECKPOINT:
    checkpoint = Path(RESUME_CHECKPOINT)
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    model = YOLO(str(checkpoint))
else:
    print("\n========== 加载模型 ==========")

    model = YOLO(PRETRAINED_WEIGHTS)
    model.info()

## 8. 开始训练并统计资源

每轮保存的 `last.pt` 位于 Drive；运行时被终止仍可能丢失当前未保存的一轮。
重复执行新训练会创建带编号的实验目录。续训时耗时/显存仅统计本次会话，累计轮数以训练器为准；环境记录会注明续训来源。

In [ ]:
print("\n========== 开始训练 ==========")

# 重置 GPU 显存峰值统计
if torch.cuda.is_available():
    torch.cuda.synchronize(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)

# 开始计时
train_start = time.perf_counter()

# 执行训练
if RESUME_CHECKPOINT:
    model.train(resume=True, data=str(DATA_YAML), device=DEVICE, workers=WORKERS)
else:
    model.train(
        data=str(DATA_YAML),

        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,

        device=DEVICE,
        workers=WORKERS,

        seed=SEED,
        deterministic=True,

        patience=PATIENCE,
        optimizer=OPTIMIZER,

        project=str(PROJECT_DIR),
        name=EXPERIMENT_NAME,
        exist_ok=False,

        save=True,
        plots=True,
        verbose=True,
    )

# 确保 GPU 操作完成
if torch.cuda.is_available():
    torch.cuda.synchronize(DEVICE)

# 训练耗时（分钟）
train_minutes = (
    time.perf_counter() - train_start
) / 60

if torch.cuda.is_available():

    gpu_peak_memory_gib = (
        torch.cuda.max_memory_reserved(DEVICE)
        / (1024 ** 3)
    )

else:
    gpu_peak_memory_gib = None

print("\n========== 训练资源统计 ==========")

print(f"训练耗时: {train_minutes:.2f} min")

if gpu_peak_memory_gib is not None:
    print(
        f"训练峰值显存: "
        f"{gpu_peak_memory_gib:.2f} GiB"
    )

trainer = model.trainer

# 实际完成的训练轮数
actual_epochs = int(trainer.epoch) + 1

# 最佳 Epoch
stopper = getattr(trainer, "stopper", None)

best_epoch = getattr(
    stopper, "best_epoch", None
)

if best_epoch is not None and best_epoch > 0:
    best_epoch = int(best_epoch)
else:
    best_epoch = ""

# 获取实际优化器
optimizer_object = getattr(
    trainer, "optimizer", None
)

if optimizer_object is not None:
    optimizer_name = type(
        optimizer_object
    ).__name__
else:
    optimizer_name = OPTIMIZER

print("\n========== 训练轮次信息 ==========")

print(f"设定 Epoch: {EPOCHS}")
print(f"实际 Epoch: {actual_epochs}")
if RESUME_CHECKPOINT:
    best_epoch = ""  # 恢复训练未还原 stopper 全程状态，避免误报最佳轮次。
print(f"最佳 Epoch: {best_epoch}")
print(f"实际优化器: {optimizer_name}")

# 续训时报告权重中的实际设置，避免台账使用 Notebook 中过期的参数。
EPOCHS = int(trainer.args.epochs)
BATCH_SIZE = int(trainer.args.batch)
IMG_SIZE = int(trainer.args.imgsz)
SEED = int(trainer.args.seed)

## 9. 加载最佳权重并统计模型复杂度

In [ ]:
print("\n========== 加载最佳模型 ==========")

best_path = Path(trainer.best).resolve()

if not best_path.is_file():
    raise FileNotFoundError(
        f"最佳模型权重不存在: {best_path}"
    )

run_dir = Path(trainer.save_dir).resolve()

print(f"最佳模型路径: {best_path}")

best_model = YOLO(str(best_path))

print("\n========== 模型复杂度 ==========")

# 参数量（百万）
parameters_M = (
        sum(
            p.numel()
            for p in best_model.model.parameters()
        ) / 1e6
)

# 计算 GFLOPs
# 注意：verbose=True 才返回模型统计信息
model_info = best_model.info(
    imgsz=IMG_SIZE,
    verbose=True
)

if (
        model_info is not None
        and len(model_info) >= 4
):
    gflops = model_info[3]
else:
    gflops = None

# GFLOPs 为 0 可能表示分析失败
if gflops is not None and gflops > 0:
    gflops = round(float(gflops), 2)
else:
    gflops = ""

print(f"模型参数量: {parameters_M:.3f} M")
print(f"GFLOPs: {gflops}")

In [ ]:
# 将实际依赖、数据配置和硬件信息随每次实验保存，便于复现。
environment = {
    "python": sys.version, "torch": torch.__version__,
    "ultralytics": ultralytics.__version__, "cuda": torch.version.cuda,
    "gpu": gpu_name, "data_root": str(DATA_ROOT),
    "resume_checkpoint": RESUME_CHECKPOINT,
    "resource_statistics_scope": "current_session",
}
(run_dir / "environment.json").write_text(json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8")
(run_dir / "requirements-freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
shutil.copy2(DATA_YAML, run_dir / "data_colab.yaml")
if "MODEL_CONFIG" in globals():
    shutil.copy2(MODEL_CONFIG, run_dir / MODEL_CONFIG.name)

## 10. 验证集评估

In [ ]:
print("\n========== 验证集评估 ==========")

val_metrics = best_model.val(
    data=str(DATA_YAML),
    split="val",

    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,

    plots=True,

    project=str(run_dir),
    name="validation",
    exist_ok=False,
)

print_metrics(val_metrics, "验证集")

## 11. 测试集评估

In [ ]:
print("\n========== 测试集评估 ==========")

test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",

    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,

    plots=True,

    project=str(run_dir),
    name="testing",
    exist_ok=False,
)

print_metrics(test_metrics, "测试集")

## 12. 保存并展示预测示例

In [ ]:
save_sample_predictions(
    model=best_model,
    run_dir=run_dir
)

for preview in sorted((run_dir / "sample_predictions").glob("*.jpg"))[:4]:
    display(Image(filename=str(preview), width=640))

## 13. 测量端到端 FPS

使用全部测试图片，预热 50 次、测量 3 次；包含读取、解码、预处理、推理和后处理，排除结果保存。
Notebook 使用文件清单确保 batch=1 生效，避免原脚本的图片列表被一次性加载；因此 FPS 与原脚本的列表输入测量值不直接等价。
FPS 受 GPU 型号、磁盘、依赖版本影响，跨设备结果不宜直接比较。本步骤可能需要较长时间。

In [ ]:
fps_mean, fps_std = benchmark_fps(
    model=best_model,
    test_dir=DATA_ROOT / "images" / "test",
    imgsz=IMG_SIZE,
    device=DEVICE,
    batch=1,
    warmup=50,
    repeats=3,
)

## 14. 保存实验台账和查看结果

台账追加到 `PROJECT_ROOT/records/experiment_ledger.csv`；重复执行本单元格会追加一行。
若 FPS 步骤失败，可先将 `fps_mean = fps_std = float("nan")` 再保存，台账会明确记录缺失值。

In [ ]:
print("\n========== 整理实验台账 ==========")

record = {

    # ===== 实验信息 =====
    "experiment_name": run_dir.name,
    "model": MODEL_NAME,
    "dataset": DATASET_NAME,

    # ===== 训练参数 =====
    "epochs": EPOCHS,
    "actual_epochs": actual_epochs,
    "best_epoch": best_epoch,

    "batch": BATCH_SIZE,
    "imgsz": IMG_SIZE,
    "optimizer": optimizer_name,
    "seed": SEED,

    # ===== 模型复杂度 =====
    "parameters_M": round(parameters_M, 3),
    "gflops": gflops,

    # ===== 硬件资源 =====
    "gpu_name": gpu_name,

    "gpu_peak_memory_gib": (
        round(gpu_peak_memory_gib, 2)
        if gpu_peak_memory_gib is not None
        else ""
    ),

    # ===== 验证集指标 =====
    "val_precision": round(
        float(val_metrics.box.mp), 4
    ),

    "val_recall": round(
        float(val_metrics.box.mr), 4
    ),

    "val_mAP50": round(
        float(val_metrics.box.map50), 4
    ),

    "val_mAP50_95": round(
        float(val_metrics.box.map), 4
    ),

    # ===== 测试集指标 =====
    "test_precision": round(
        float(test_metrics.box.mp), 4
    ),

    "test_recall": round(
        float(test_metrics.box.mr), 4
    ),

    "test_mAP50": round(
        float(test_metrics.box.map50), 4
    ),

    "test_mAP50_95": round(
        float(test_metrics.box.map), 4
    ),

    # ===== 训练时间 =====
    "train_minutes": round(
        train_minutes, 2
    ),

    # ===== 推理 FPS =====
    "fps_mean": round(
        fps_mean, 3
    ),

    "fps_std": round(
        fps_std, 3
    ),
}

save_experiment_record(
    record=record,
    project_root=PROJECT_ROOT
)

print("\n====================================")
print(f"{MODEL_NAME} 实验全部完成！")
print("====================================")

print(f"最佳模型: {best_path}")
print(f"最佳 Epoch: {best_epoch}")
print(f"模型参数量: {parameters_M:.3f} M")
print(f"GFLOPs: {gflops}")
print(f"训练耗时: {train_minutes:.2f} min")

print(f"实验目录: {run_dir}")

In [ ]:
for filename in ("results.png", "confusion_matrix.png"):
    path = run_dir / filename
    if path.is_file():
        display(Image(filename=str(path), width=900))
print("训练目录:", run_dir)
print("最佳权重:", best_path)
print("实验台账:", PROJECT_ROOT / "records" / "experiment_ledger.csv")

## 15. 可选：下载最佳权重

Drive 中已经保存结果。需要下载到电脑时，把开关设为 `True` 并运行。

In [ ]:
DOWNLOAD_BEST = False
if DOWNLOAD_BEST:
    from google.colab import files
    files.download(str(best_path))

## 运行提示与参考

- 每个 Notebook 可独立运行；不要在同一内核中交错执行多个实验的单元格。
- 断线后若内存变量消失，先重新执行安装、挂载与数据准备。`/content` 数据可能已经丢失，需要重新复制/解压。
- 不使用 Drive 时，结果位于临时运行时磁盘，请自行及时下载。
- 本文件未包含训练输出，未在当前机器启动完整训练。默认参数来源于脚本，而非旧 `runs/args.yaml`。

官方参考：[Colab FAQ](https://research.google.com/colaboratory/faq.html) · [Ultralytics 训练与恢复](https://docs.ultralytics.com/modes/train/)